# Batch cell typing – IHOPE project

Runs normalization, AnnData construction, GMM thresholding, rule-based cell typing, and summary export for all 14 samples. Starts from pre-filtered CSVs. Paths are built with pathlib so this runs the same on Windows or Mac.

In [ ]:
import sys
import gc
from pathlib import Path
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "processed"
ANNDATA_DIR = DATA_DIR / "anndata"
ANNDATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from scripts.anndata_helpers import load_and_build_anndata, save_h5ad
from scripts.annotation import compute_positivity_matrix
from scripts.celltype_rules_IHOPE import assign_cell_types_bool_IHOPE
from scripts.summary_celltypes_IHOPE import summarize_celltypes_IHOPE

## Reload transforms

Reloading the module alone is not enough. The import line must be rerun as well, or a stale binding of `apply_transform` persists.

In [ ]:
import importlib
from scripts import transforms
importlib.reload(transforms)
from scripts.transforms import apply_transform

## Sample list

`file_basename` plus `input_suffix` is the name used to find the input CSV on disk. `out_basename` is the name used for all outputs. They differ where files have been renamed or used a non-standard suffix.

In [ ]:
# (file_basename, input_suffix, out_basename)
# input_suffix is either "_cleaned_filtered" or "_cleaned_filtered_looped"
SAMPLES = [
    ("IHOPE14_MedLN_BottomLeft",  "_cleaned_filtered",        "IHOPE14_MedLN_BottomLeft"),
    ("IHOPE14_MedLN_TopRight",    "_cleaned_filtered_looped", "IHOPE14_MedLN_TopRight"),
    ("IHOPE14_MedLN_BottomRight", "_cleaned_filtered_looped", "IHOPE14_MedLN_BottomRight"),
    ("IHOPE14_mesLN",             "_cleaned_filtered_looped", "IHOPE14_MesLN"),
    ("IHOPE20_LN",                "_cleaned_filtered",        "IHOPE20_MedLN"),
    ("IHOPE20_Spleen",            "_cleaned_filtered_looped", "IHOPE20_Spleen"),
    ("IHOPE26_LN",                "_cleaned_filtered",        "IHOPE26_MedLN"),
    ("IHOPE26_Spleen",            "_cleaned_filtered",        "IHOPE26_Spleen"),
    ("IHOPE27_LN",                "_cleaned_filtered",        "IHOPE27_MedLN"),
    ("IHOPE27_Spleen",            "_cleaned_filtered",        "IHOPE27_Spleen"),
    ("IHOPE39_LN",                "_cleaned_filtered",        "IHOPE39_MedLN"),
    ("IHOPE39_MesLN_A",           "_cleaned_filtered",        "IHOPE39_MesLN_A"),
    ("IHOPE39_MesLN_B",           "_cleaned_filtered",        "IHOPE39_MesLN_B"),
    ("IHOPE39_Spleen",            "_cleaned_filtered",        "IHOPE39_Spleen"),
]

## Batch pipeline

For each sample: normalize from the cleaned filtered CSV, build AnnData, run GMM thresholding, assign cell types, and export a summary CSV. Intermediate objects are freed as early as possible to keep peak memory low.

In [ ]:
failed = []

for file_basename, input_suffix, out_basename in SAMPLES:
    print(f"Processing: {out_basename}")

    try:
        # Normalize
        input_file  = DATA_DIR / f"{file_basename}{input_suffix}.csv"
        zscore_file = DATA_DIR / f"{out_basename}_cleaned_filtered_log2_zscore.csv"

        df_out, markers, metadata, fig_raw, fig_normalized = apply_transform(
            input_file=str(input_file),
            method="zscore",
            output_file=str(zscore_file),
            save_plot=True,
        )
        print(f"  Normalized: {len(df_out)} cells, {len(markers)} markers")

        # Free the dataframe and figures right away. load_and_build_anndata reads
        # the CSV back from disk, so df_out is not needed past this point.
        plt.close(fig_raw)
        plt.close(fig_normalized)
        del df_out, markers, metadata, fig_raw, fig_normalized
        plt.close('all')
        gc.collect()

        # Build AnnData
        adata = load_and_build_anndata(str(zscore_file))
        print(f"  AnnData built: {adata.shape}")

        # GMM thresholding
        adata, thresholds, best_gmms = compute_positivity_matrix(
            adata,
            quantile=0.8,
            random_state=0,
        )
        print(f"  GMM thresholding done")
        del thresholds, best_gmms
        gc.collect()

        # Save AnnData after GMM
        h5ad_gmm = ANNDATA_DIR / f"{out_basename}_filtered_log2_zscore_GMM.h5ad"
        save_h5ad(adata, str(h5ad_gmm))
        print(f"  Saved: {h5ad_gmm}")

        # Rule-based cell typing
        adata = assign_cell_types_bool_IHOPE(adata)
        print(f"  Cell types assigned")

        # Save AnnData with cell types
        h5ad_typed = ANNDATA_DIR / f"{out_basename}_filtered_log2_zscore_GMM_IHOPE_celltypes.h5ad"
        save_h5ad(adata, str(h5ad_typed))
        print(f"  Saved: {h5ad_typed}")

        # Summary CSV
        df_summary = summarize_celltypes_IHOPE(
            adata,
            filename=f"{out_basename}_filtered_log2_zscore_IHOPE_summary.csv",
        )
        print(f"  Summary exported")

        del adata, df_summary
        gc.collect()

    except Exception as e:
        print(f"  ERROR: {e}")
        failed.append((out_basename, str(e)))

print("\nBatch complete.")
if failed:
    print(f"Failed samples ({len(failed)}):")
    for name, err in failed:
        print(f"  {name}: {err}")
else:
    print("All samples processed successfully.")

**On normalized data (with BANKSY information)**

In [14]:
import importlib
import scripts.celltype_rules_IHOPE
importlib.reload(scripts.celltype_rules_IHOPE)

from scripts.celltype_rules_IHOPE import (
    assign_cell_types_bool_IHOPE,
    add_TfH_like_cells,
    add_spatial_B_context,
)

In [15]:
import gc
from pathlib import Path
import scanpy as sc

BANKSY_DIR = PROJECT_ROOT / "data" / "processed" / "anndata" / "zscore"
CELLTYPED_DIR = BANKSY_DIR / "celltyped"
CELLTYPED_DIR.mkdir(parents=True, exist_ok=True)

input_files = sorted(BANKSY_DIR.glob("*_banksy.h5ad"))

failed = []

for input_file in input_files:
    print(f"Processing: {input_file.name}")

    try:
        output_file = CELLTYPED_DIR / f"{input_file.stem}_celltypes.h5ad"

        adata = sc.read_h5ad(str(input_file))
        adata = assign_cell_types_bool_IHOPE(adata)
        adata.write_h5ad(str(output_file))
        print(f"  Saved: {output_file.name}")

        del adata
        gc.collect()

    except Exception as e:
        print(f"  ERROR: {e}")
        failed.append((input_file.name, str(e)))

print("Batch complete.")
if failed:
    print(f"Failed samples ({len(failed)})")
    for name, err in failed:
        print(f"  {name}: {err}")
else:
    print("All samples processed.")

Processing: IHOPE14_MedLN_BottomLeft_filtered_log2_zscore_banksy.h5ad

[TYPE VALIDATION]
Counts of assignments per cell:
1    112174
Name: count, dtype: int64
TYPE assignment is strictly exclusive (each cell has exactly one type)
Total cells: 112174

Level — type:
  type_B: 6420 (5.7% of total)
  type_T: 27795 (24.8% of total)
  type_NK: 43 (0.0% of total)
  type_Myeloid: 13012 (11.6% of total)
  type_Stromal: 12153 (10.8% of total)
  type_Endothelial: 15652 (14.0% of total)
  type_unclassified: 37099 (33.1% of total)

Level — intermediate:
  intermediate_CD4_T: 14652 (13.1% of total)
  intermediate_CD8_T: 4051 (3.6% of total)
  intermediate_T_naive: 692 (0.6% of total)
  intermediate_T_memory: 13203 (11.8% of total)
  intermediate_B_naive: 125 (0.1% of total)
  intermediate_B_memory: 2497 (2.2% of total)

Level — subtype:
  subtype_B_naive: 125 (0.1% of total)
  subtype_B_GC: 630 (0.6% of total)
  subtype_B_Plasmablast: 1643 (1.5% of total)
  subtype_TN_CD4: 519 (0.5% of total)
  subt

Just summary: